In [12]:
# Celda 1: Instalación de paquetes
%pip install azure-search-documents azure-identity dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Celda 2: Importaciones
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SemanticSearch
)

In [2]:
# Celda 3: Configuración
load_dotenv()
service_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
admin_key = os.getenv("AZURE_SEARCH_KEY")
index_name = "ponal-userdocs-index"
vector_dimensions = 3072

In [3]:
# Celda 4: Definición del índice
fields = [
    # ID principal
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        searchable=False
    ),
    
    # ID del chunk
    SimpleField(
        name="chunk_id",
        type=SearchFieldDataType.String,
        searchable=False
    ),
    
    # Nombre del archivo
    SearchableField(
        name="filename",
        type=SearchFieldDataType.String,
        searchable=True,
        filterable=True
    ),
    
    # Contenido del documento
    SearchableField(
        name="content",
        type=SearchFieldDataType.String,
        searchable=True,
        analyzer_name="es.microsoft"
    ),
    
    # Embedding vectorial
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=vector_dimensions,
        vector_search_profile_name="my-vector-profile"
    ),
    
    # ID de usuario
    SimpleField(
        name="user_id",
        type=SearchFieldDataType.String,
        filterable=True,
        searchable=False
    ),
    
    # ID de sesión
    SimpleField(
        name="session_id",
        type=SearchFieldDataType.String,
        filterable=True,
        searchable=False
    ),
    
    # ID del archivo
    SimpleField(
        name="file_id",
        type=SearchFieldDataType.String,
        filterable=True
    ),
    
    # URL del blob
    SimpleField(
        name="blob_url",
        type=SearchFieldDataType.String,
        searchable=False
    ),
    
    # Páginas (como string para rangos como "1-3,5")
    SimpleField(
        name="pages",
        type=SearchFieldDataType.Int32,
        searchable=False
    ),
    
    # Fecha de creación
    SimpleField(
        name="created_at",
        type=SearchFieldDataType.DateTimeOffset,
        filterable=True,
        sortable=True
    )
]

# Configurar algoritmo HNSW para búsqueda vectorial
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="my-hnsw",
            parameters={
                "m": 4,
                "efConstruction": 400,
                "metric": "cosine"
            }
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="my-vector-profile",
            algorithm_configuration_name="my-hnsw"
        )
    ]
)

# Configurar búsqueda semántica
semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="filename"),
        content_fields=[SemanticField(field_name="content")]
    )
)

semantic_search = SemanticSearch(configurations=[semantic_config])

# Crear la definición del índice
index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search
)

print("Índice definido correctamente")
print(f"Nombre del índice: {index_name}")
print(f"Número de campos: {len(fields)}")
print(f"Dimensiones del vector: {vector_dimensions}")

Índice definido correctamente
Nombre del índice: ponal-userdocs-index
Número de campos: 11
Dimensiones del vector: 3072


In [4]:
# Celda 5: Creación del índice
try:
    # Crear cliente para gestión de índices
    credential = AzureKeyCredential(admin_key)
    index_client = SearchIndexClient(endpoint=service_endpoint, credential=credential)
    
    # Crear el índice
    print(f"Creando índice '{index_name}' en Azure Cognitive Search...")
    result = index_client.create_index(index)
    print(f"✅ Índice '{index_name}' creado exitosamente!")
    
    # Mostrar información del índice creado
    print(f"\nInformación del índice creado:")
    print(f"- Nombre: {result.name}")
    print(f"- Campos: {len(result.fields)}")
    print(f"- Búsqueda vectorial: {'Habilitada' if result.vector_search else 'Deshabilitada'}")
    print(f"- Búsqueda semántica: {'Habilitada' if result.semantic_search else 'Deshabilitada'}")
    
except Exception as e:
    print(f"❌ Error al crear el índice: {str(e)}")
    print("Verifica las credenciales y conexión con Azure Cognitive Search")

Creando índice 'ponal-userdocs-index' en Azure Cognitive Search...
✅ Índice 'ponal-userdocs-index' creado exitosamente!

Información del índice creado:
- Nombre: ponal-userdocs-index
- Campos: 11
- Búsqueda vectorial: Habilitada
- Búsqueda semántica: Habilitada
